In [17]:
pip install --quiet tensorflow

I0716 11:56:03.648219    4783 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Note: you may need to restart the kernel to use updated packages.


In [18]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [19]:
print(tf.__version__)

2.21.0


In [8]:
BUCKET ="gs://bucket-for-kpf-project"
MODEL_SAVE ="gs://bucket-for-kpf-project/tst"
DATA_URI="gs://bucket-for-kpf-project/test_UCI_Credit_Card.csv"
df = pd.read_csv(DATA_URI)

In [9]:
train_dataset = df.sample(frac=0.8,random_state=0)
test_dataset = df.drop(train_dataset.index)
train_stats = train_dataset.describe()
train_stats.pop("default.payment.next.month")
train_stats = train_stats.transpose()

In [10]:
train_labels = train_dataset.pop('default.payment.next.month')
test_labels = test_dataset.pop('default.payment.next.month')

In [11]:
def norm(x):
  return (x - train_stats['mean']) / train_stats['std']
normed_train_data = norm(train_dataset)
normed_test_data = norm(test_dataset)

In [10]:
def build_model():

  model = Sequential()
  model.add(Dense(256, activation='relu', input_shape=(train_dataset.shape[1],)))  # Input layer
  model.add(Dense(256, activation='relu'))  # Hidden layer
  model.add(Dense(1, activation='sigmoid'))  # Output layer (sigmoid for binary classification)

  model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

  return model

model = build_model()

/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1784138973.711918    4288 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [11]:
model.fit(normed_train_data, train_labels, 
                    epochs=10, validation_split = 0.2)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8167 - loss: 0.4601 - val_accuracy: 0.8135 - val_loss: 0.4581
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8212 - loss: 0.4341 - val_accuracy: 0.8158 - val_loss: 0.4529
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8251 - loss: 0.4272 - val_accuracy: 0.8125 - val_loss: 0.4605
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8245 - loss: 0.4247 - val_accuracy: 0.8163 - val_loss: 0.4535
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8254 - loss: 0.4204 - val_accuracy: 0.8152 - val_loss: 0.4509
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8271 - loss: 0.4169 - val_accuracy: 0.8167 - val_loss: 0.4516
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8282 - loss: 0.4149 - val_accuracy: 0.8142 - val_loss: 0.4585
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8273 - loss: 0.4116 - val_accuracy: 0.

In [13]:
loss, acc = new_model.evaluate(normed_test_data, test_labels, verbose=0)

In [15]:
print("test_accuracy:", acc, type(acc))

test_accuracy: 0.8069999814033508 <class 'float'>


In [17]:
model.save('gs://custom-trin-bucket/custom_model_test1.keras')

In [12]:
new_model = tf.keras.models.load_model('gs://custom-trin-bucket/custom_model_test1.keras')

In [5]:
new_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │         6,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 217,349 (849.02 KB)

 Trainable params: 72,449 (283.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 144,900 (566.02 KB)